In [ ]:
res = []
for _, r in patients[patients.split == "dev"].iterrows():
    p = next((LUNA / r.subset).rglob(f"{r.seriesuid}.mhd"))
    im = resample_isotropic(sitk.ReadImage(str(p)), 1.0)
    m = segment_lungs(sitk.GetArrayFromImage(im))
    v = nodule_voxels(im, ann[ann.seriesuid == r.seriesuid])
    res.append({"seriesuid": r.seriesuid, "volume_L": m.sum() / 1e6,
                "nodules": len(v), "dans_masque": nodules_inside(m, v)})
res = pd.DataFrame(res)
res.to_csv(ROOT / "results" / "segmentation_eval.csv", index=False)
print(res)
print("Nodules retrouvés :", res.dans_masque.sum(), "/", res.nodules.sum())

In [ ]:
nod = ann[ann.seriesuid == row.seriesuid]
vox = nodule_voxels(img_rs, nod)
print(f"{nodules_inside(mask, vox)} nodules sur {len(vox)} dans le masque")

In [ ]:
zs = np.where(mask.any(axis=(1, 2)))[0]
idx = np.linspace(zs.min() + 10, zs.max() - 10, 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for ax, k in zip(axes.ravel(), idx):
    ax.imshow(vol[k], cmap="gray", vmin=-1000, vmax=400)
    ax.contour(mask[k].astype(float), levels=[0.5], colors="r", linewidths=1)
    ax.set_title(f"z = {k}"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
t0 = time.time()
mask = segment_lungs(vol, thr=-320, r_open=2, r_close=8)
print(f"{time.time() - t0:.0f} s, volume pulmonaire : {mask.sum() / 1e6:.2f} L")

In [ ]:
row = patients[patients.split == "dev"].iloc[0]
path = next((LUNA / row.subset).rglob(f"{row.seriesuid}.mhd"))
img = sitk.ReadImage(str(path))
img_rs = resample_isotropic(img, 1.0)
vol = sitk.GetArrayFromImage(img_rs)
print("avant :", img.GetSize(), img.GetSpacing())
print("après :", img_rs.GetSize(), img_rs.GetSpacing())

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import SimpleITK as sitk

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.segmentation import resample_isotropic, segment_lungs, nodule_voxels, nodules_inside

LUNA = ROOT / "data" / "luna16"
patients = pd.read_csv(ROOT / "results" / "patients.csv")
ann = pd.read_csv(LUNA / "annotations.csv")